In [2]:
from azureml.core import Workspace, Model

ws = Workspace.from_config()

registered_model = Model.register(
    workspace=ws,
    model_path="g10tfm_model_test1.pkl",
    model_name="g10tfm-model-v2"
)

print("Modelo registrado:", registered_model.name, registered_model.version)


Registering model g10tfm-model-v2
Modelo registrado: g10tfm-model-v2 4


In [3]:
from azureml.core.environment import Environment
from azureml.core.model import InferenceConfig

env = Environment.from_conda_specification(
    name="g10tfm-env-v2",
    file_path="env.yml"
)

inference_config = InferenceConfig(
    entry_script="score_v2.py",
    environment=env
)


Warning, azureml-defaults not detected in provided environment pip dependencies. The azureml-defaults package contains requirements for the inference stack to run, and should be included.


In [4]:
from azureml.core.webservice import AciWebservice

aci_config = AciWebservice.deploy_configuration(
    cpu_cores=1,
    memory_gb=1,
    auth_enabled=True
)


In [5]:
service_name = "g10tfm-endpoint-v2"

service = Model.deploy(
    workspace=ws,
    name=service_name,
    models=[registered_model],
    inference_config=inference_config,
    deployment_config=aci_config,
    overwrite=True
)

service.wait_for_deployment(show_output=True)

/tmp/ipykernel_3586/712076857.py:3: FutureWarning: azureml.core.model:
To leverage new model deployment capabilities, AzureML recommends using CLI/SDK v2 to deploy models as online endpoint, 
please refer to respective documentations 
https://docs.microsoft.com/azure/machine-learning/how-to-deploy-managed-online-endpoints /
https://docs.microsoft.com/azure/machine-learning/how-to-attach-kubernetes-anywhere 
For more information on migration, see https://aka.ms/acimoemigration 
To disable CLI/SDK v1 deprecation warning set AZUREML_LOG_DEPRECATION_WARNING_ENABLED to 'False'
  service = Model.deploy(


Tips: You can try get_logs(): https://aka.ms/debugimage#dockerlog or local deployment: https://aka.ms/debugimage#debug-locally to debug if deployment takes longer than 10 minutes.
Running
2025-12-01 23:46:05+00:00 Registering the environment.
2025-12-01 23:46:05+00:00 Generating deployment configuration.
2025-12-01 23:46:07+00:00 Submitting deployment to compute.
Failed


Service deployment polling reached non-successful terminal state, current service state: Transitioning
Operation ID: 4ebed4be-8952-4924-a111-baeeb1d8817e
More information can be found using '.get_logs()'
Error:
{
  "code": "AuthorizationFailed",
  "statusCode": 403,
  "message": "ACI Service request failed. Reason: The client '3e598908-4565-46da-b24d-20a30ef1b8a4' with object id '7f622c5e-bf9e-4ed3-a265-4f45d30561e5' does not have authorization to perform action 'Microsoft.ContainerInstance/containerGroups/write' over scope '/subscriptions/506e4ca3-b9db-4960-aedf-c02bf0ea7e28/resourceGroups/jaragono-rg/providers/Microsoft.ContainerInstance/containerGroups/g10tfm-endpoint-v2-d6bSwxABfUS7tBUi3JF7yg' or the scope is invalid. If access was recently granted, please refresh your credentials.."
}



WebserviceException: WebserviceException:
	Message: Service deployment polling reached non-successful terminal state, current service state: Transitioning
Operation ID: 4ebed4be-8952-4924-a111-baeeb1d8817e
More information can be found using '.get_logs()'
Error:
{
  "code": "AuthorizationFailed",
  "statusCode": 403,
  "message": "ACI Service request failed. Reason: The client '3e598908-4565-46da-b24d-20a30ef1b8a4' with object id '7f622c5e-bf9e-4ed3-a265-4f45d30561e5' does not have authorization to perform action 'Microsoft.ContainerInstance/containerGroups/write' over scope '/subscriptions/506e4ca3-b9db-4960-aedf-c02bf0ea7e28/resourceGroups/jaragono-rg/providers/Microsoft.ContainerInstance/containerGroups/g10tfm-endpoint-v2-d6bSwxABfUS7tBUi3JF7yg' or the scope is invalid. If access was recently granted, please refresh your credentials.."
}
	InnerException None
	ErrorResponse 
{
    "error": {
        "message": "Service deployment polling reached non-successful terminal state, current

In [ ]:
service.wait_for_deployment(show_output=True)

In [9]:
from azureml.core.webservice import AksWebservice

deployment_config = AksWebservice.deploy_configuration(
    cpu_cores=1,
    memory_gb=4,
    enable_app_insights=True
)

service_name = "g10tfm-endpoint-v2"

service = Model.deploy(
    workspace=ws,
    name=service_name,
    models=[registered_model],
    inference_config=inference_config,
    deployment_config=deployment_config,  # ← ESTE es el importante
    deployment_target=cluster,           # ← Tu cluster g10tfm-cluster
    overwrite=True
)

service.wait_for_deployment(show_output=True)


/tmp/ipykernel_3586/3560368511.py:11: FutureWarning: azureml.core.model:
To leverage new model deployment capabilities, AzureML recommends using CLI/SDK v2 to deploy models as online endpoint, 
please refer to respective documentations 
https://docs.microsoft.com/azure/machine-learning/how-to-deploy-managed-online-endpoints /
https://docs.microsoft.com/azure/machine-learning/how-to-attach-kubernetes-anywhere 
For more information on migration, see https://aka.ms/acimoemigration 
To disable CLI/SDK v1 deprecation warning set AZUREML_LOG_DEPRECATION_WARNING_ENABLED to 'False'
  service = Model.deploy(


WebserviceException: WebserviceException:
	Message: Cannot switch between compute types when overwriting a service. Please delete existing service first.
	InnerException None
	ErrorResponse 
{
    "error": {
        "message": "Cannot switch between compute types when overwriting a service. Please delete existing service first."
    }
}

In [10]:
from azureml.core.webservice import Webservice

service_name = "g10tfm-endpoint-v2"
service = Webservice(name=service_name, workspace=ws)
service.delete()
print("Endpoint eliminado")


Running
2025-12-02 00:00:26+00:00 Deleting service.
2025-12-02 00:00:27+00:00 Deleting service entity.
Succeeded
Endpoint eliminado


In [12]:
service = Model.deploy(
    workspace=ws,
    name="g10tfm-endpoint-v2",
    models=[registered_model],
    inference_config=inference_config,
    deployment_config=deployment_config,  
    deployment_target=cluster       
)

service.wait_for_deployment(show_output=True)


/tmp/ipykernel_3586/1421573630.py:1: FutureWarning: azureml.core.model:
To leverage new model deployment capabilities, AzureML recommends using CLI/SDK v2 to deploy models as online endpoint, 
please refer to respective documentations 
https://docs.microsoft.com/azure/machine-learning/how-to-deploy-managed-online-endpoints /
https://docs.microsoft.com/azure/machine-learning/how-to-attach-kubernetes-anywhere 
For more information on migration, see https://aka.ms/acimoemigration 
To disable CLI/SDK v1 deprecation warning set AZUREML_LOG_DEPRECATION_WARNING_ENABLED to 'False'
  service = Model.deploy(


WebserviceException: WebserviceException:
	Message: Unsupported ClusterType BatchAI for the Compute resource Id 'g10tfm-cluster'.
	InnerException None
	ErrorResponse 
{
    "error": {
        "message": "Unsupported ClusterType BatchAI for the Compute resource Id 'g10tfm-cluster'."
    }
}